In [1]:
# Importing the libraries

from dotenv import load_dotenv 
from anthropic import Anthropic
import json

In [2]:
load_dotenv()

True

In [3]:
client = Anthropic()
model = "claude-haiku-4-5"

In [6]:
# Helper functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [9]:
# Dataset Creation Function

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate a comrehensive breakdown of the task at hand into further subtasks.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a 4-5 sub tasks.
* Focus on tasks that are comonly asked to any productivity agent.

Please generate 10 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [10]:
# Calling the generate dataset function

dataset = generate_dataset()
print(dataset)

[{'task': 'Organize a team meeting to discuss Q4 project roadmap and budget allocation'}, {'task': 'Create a comprehensive marketing campaign for a new product launch'}, {'task': 'Develop an onboarding process for new employees in the engineering department'}, {'task': 'Plan and execute a company-wide cost reduction initiative'}, {'task': 'Prepare a quarterly business review presentation for executive stakeholders'}, {'task': 'Set up a customer feedback collection and analysis system'}, {'task': 'Design and implement a remote work policy for the organization'}, {'task': 'Create a project timeline and resource allocation plan for a 6-month software development initiative'}, {'task': "Establish a knowledge management system for the company's documentation and best practices"}, {'task': 'Develop a performance evaluation framework and conduct mid-year reviews for the sales team'}]


In [11]:
# Saves the generated dataset in the dataset.json file

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [12]:
# Defining the run prompt function that runs the our prompt we need to evaluate 

def run_prompt(test_case):                 # Test case refers to the test dataset generated above
    """Merges the prompt and the test case input and returns the result"""
    prompt = f"""
    Break the following task into further sub tasks.
    
    {test_case["task"]}
    """
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    add_assistant_message(messages, output)
    return output

In [14]:
# Defining the run_test_case funtion that runs the test cases

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [22]:
# Defining the function that runs the avaluation process and returns the result 

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

In [23]:
# To execute our evaluation pipeline, we load our dataset and run it through our functions:

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [24]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Sub-tasks for Organizing Q4 Project Roadmap & Budget Meeting\n\n## Planning & Preparation\n- [ ] Define meeting objectives and desired outcomes\n- [ ] Determine required attendees and stakeholders\n- [ ] Identify any pre-read materials or reports needed\n- [ ] Gather current project status and budget data\n\n## Scheduling & Logistics\n- [ ] Check attendees' calendar availability\n- [ ] Book meeting room or video conference platform\n- [ ] Send calendar invitations with meeting details\n- [ ] Choose appropriate meeting duration (suggest 1.5-2 hours)\n\n## Content Development\n- [ ] Create agenda with time allocations for each topic\n- [ ] Prepare presentation slides or documents on Q4 roadmap\n- [ ] Compile current budget status and allocation options\n- [ ] Identify key discussion points and decision items\n\n## Pre-Meeting Communication\n- [ ] Share agenda with attendees 3-5 days in advance\n- [ ] Distribute background materials or reports early\n- [ ] Request i

In [25]:
# Saves the generated results in the results.json file

with open('results.json', 'w') as f:
    json.dump(results, f, indent=2)